<a target="_blank" href="https://colab.research.google.com/github/cesarschoollectures/am-labs/blob/main/assignments/E01_Decision_Tree.ipynb">
<img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Aprendizado de Máquina

Nesta atividade, você irá trabalhar com o dataset Fashion MNIST utilizando modelos de classificação do sklearn.

O foco NÃO é apenas obter bons resultados, mas garantir que o experimento seja:
- correto
- reprodutível
- bem estruturado
- criticamente analisado

# Dicas importantes

## Sobre o dataset (Fashion MNIST)

- Utilize `fetch_openml` do sklearn para carregar os dados
- Use: `as_frame=False`
- Use: `mnist_784`
- Converta os rótulos para inteiro:
  
  ```python
  y = y.astype(int)
  ```

# Questão 1

Implemente uma função load_data(seed) que:

Carregue o dataset `Fashion MNIST`
Realize a separação em treino e teste
Utilize `train_test_split` com controle de aleatoriedade
Retorne: `X_train`, `X_test`, `y_train`, `y_test`

Depois responda: 
É necessário normalizar os dados para esse tipo de modelo? Justifique.

**Solução**:

In [ ]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

def load_data(seed=42):
    X, y = fetch_openml("mnist_784", version=1, as_frame=False, return_X_y=True)
    y = y.astype(int)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y
    )

    return X_train, X_test, y_train, y_test

# Questão 2

Implemente as funções:

`train_random_forest(X_train, y_train, seed)`
`train_adaboost(X_train, y_train, seed)`

## Requisitos:

Utilizar os modelos do `sklearn`
Garantir reprodutibilidade com `random_state`

**Solução**:

In [ ]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

def train_random_forest(X_train, y_train, seed=42):
    model = RandomForestClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

def train_adaboost(X_train, y_train, seed=42):
    model = AdaBoostClassifier(random_state=seed)
    model.fit(X_train, y_train)
    return model

# Questão 3

Implemente a função:

- `evaluate(model, X_test, y_test)`

Ela deve:
- Realizar predições
- Retornar a acurácia do modelo

**Solução**:

In [13]:
from sklearn.metrics import accuracy_score

def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return accuracy_score(y_test, y_pred)

A função evaluate realiza a predição no conjunto de teste e calcula a acurácia comparando os rótulos previstos com os rótulos reais.

Com isso, ela retorna uma medida direta de desempenho do modelo em dados não vistos durante o treino, mantendo o fluxo do experimento simples, correto e reprodutível.

# Questão 4

Implemente a função:

- `run_pipeline(model_type="rf", seed=42)`

Ela deve:
- Carregar os dados
- Treinar o modelo escolhido (`rf` ou `ab`)
- Avaliar o modelo
- Retornar a acurácia

**Solução**:

In [4]:
def run_pipeline(model_type="rf", seed=42):
    X_train, X_test, y_train, y_test = load_data(seed)

    if model_type == "rf":
        model = train_random_forest(X_train, y_train, seed)
    elif model_type == "ab":
        model = train_adaboost(X_train, y_train, seed)
    else:
        raise ValueError("model_type deve ser 'rf' ou 'ab'")

    acc = evaluate(model, X_test, y_test)
    return acc

**Em qual profundidade começa o overfitting?**
**Por que a árvore consegue 100% no treino quando max_depth=None?**

# Questão 5

Execute o pipeline para ambos os modelos:

- Random Forest
- AdaBoost

## Apresente:
- Acurácia, Precisão, Recall e F1-Score de cada modelo

## Responda:
- Qual modelo apresentou melhor desempenho inicial?

**Solução**:

In [22]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_metrics(model, X_test, y_test):
    y_pred = model.predict(X_test)
    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "f1": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    }

seed = 42
X_train, X_test, y_train, y_test = load_data(seed)

rf_model = train_random_forest(X_train, y_train, seed)
ab_model = train_adaboost(X_train, y_train, seed)

rf_metrics = evaluate_metrics(rf_model, X_test, y_test)
ab_metrics = evaluate_metrics(ab_model, X_test, y_test)

print("Random Forest:")
for metric_name, metric_value in rf_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

print("\nAdaBoost:")
for metric_name, metric_value in ab_metrics.items():
    print(f"{metric_name}: {metric_value:.4f}")

best_model = "Random Forest" if rf_metrics["f1"] >= ab_metrics["f1"] else "AdaBoost"
print(f"\nMelhor desempenho inicial (por F1): {best_model}")

Random Forest:
accuracy: 0.9661
precision: 0.9661
recall: 0.9661
f1: 0.9661

AdaBoost:
accuracy: 0.6681
precision: 0.6884
recall: 0.6681
f1: 0.6711

Melhor desempenho inicial (por F1): Random Forest


# Questão 6

Execute o pipeline utilizando diferentes seeds (ex: 42 e 7).

## Analise:
- Os resultados mudaram?

## Responda:
- O experimento é reprodutível? Justifique.

**Solução**:

Os resultados **mudaram** entre as seeds 42 e 7. Exemplo pelo F1:
- Random Forest: 0.9661 (seed 42) vs 0.9682 (seed 7), variação pequena (0.0021).
- AdaBoost: 0.6711 (seed 42) vs 0.6160 (seed 7), variação maior (0.0551).

Portanto, o experimento é **reprodutível** quando a seed é fixada (mesmo `random_state` gera o mesmo resultado), mas os valores podem variar ao trocar a seed, o que é esperado por causa da aleatoriedade no particionamento e no treinamento.

In [17]:
seeds = [42, 7]
results = {}

for seed in seeds:
    X_train, X_test, y_train, y_test = load_data(seed)

    rf_model = train_random_forest(X_train, y_train, seed)
    ab_model = train_adaboost(X_train, y_train, seed)

    results[seed] = {
        "Random Forest": evaluate_metrics(rf_model, X_test, y_test),
        "AdaBoost": evaluate_metrics(ab_model, X_test, y_test),
    }

for seed in seeds:
    print(f"\nSeed {seed}")
    for model_name, metrics in results[seed].items():
        print(f"{model_name}:")
        print(
            f"  accuracy={metrics['accuracy']:.4f}, "
            f"precision={metrics['precision']:.4f}, "
            f"recall={metrics['recall']:.4f}, "
            f"f1={metrics['f1']:.4f}"
        )

print("\nComparação entre seeds (mudança absoluta no F1):")
for model_name in ["Random Forest", "AdaBoost"]:
    f1_42 = results[42][model_name]["f1"]
    f1_7 = results[7][model_name]["f1"]
    diff = abs(f1_42 - f1_7)
    print(f"{model_name}: |F1(42) - F1(7)| = {diff:.4f}")

print("\nConclusão:")
print(
    "Os resultados podem variar entre seeds diferentes porque o particionamento e o treino mudam. "
    "Ainda assim, com seed fixa, o experimento é reprodutível (mesmas funções + mesmo random_state => mesmos resultados)."
)


Seed 42
Random Forest:
  accuracy=0.9661, precision=0.9661, recall=0.9661, f1=0.9661
AdaBoost:
  accuracy=0.6681, precision=0.6884, recall=0.6681, f1=0.6711

Seed 7
Random Forest:
  accuracy=0.9682, precision=0.9682, recall=0.9682, f1=0.9682
AdaBoost:
  accuracy=0.6115, precision=0.6653, recall=0.6115, f1=0.6160

Comparação entre seeds (mudança absoluta no F1):
Random Forest: |F1(42) - F1(7)| = 0.0021
AdaBoost: |F1(42) - F1(7)| = 0.0551

Conclusão:
Os resultados podem variar entre seeds diferentes porque o particionamento e o treino mudam. Ainda assim, com seed fixa, o experimento é reprodutível (mesmas funções + mesmo random_state => mesmos resultados).


# Questão 7

Para pelo menos um dos modelos:

- Compare a acurácia em treino e teste

## Responda:
- Existe overfitting?
- Qual modelo tende a sofrer mais com isso?

In [20]:
from sklearn.metrics import accuracy_score

seed = 42
X_train, X_test, y_train, y_test = load_data(seed)

rf_model = train_random_forest(X_train, y_train, seed)
ab_model = train_adaboost(X_train, y_train, seed)

rf_train_acc = accuracy_score(y_train, rf_model.predict(X_train))
rf_test_acc = accuracy_score(y_test, rf_model.predict(X_test))
rf_gap = rf_train_acc - rf_test_acc

ab_train_acc = accuracy_score(y_train, ab_model.predict(X_train))
ab_test_acc = accuracy_score(y_test, ab_model.predict(X_test))
ab_gap = ab_train_acc - ab_test_acc

print("Random Forest")
print(f"  Acurácia treino: {rf_train_acc:.4f}")
print(f"  Acurácia teste : {rf_test_acc:.4f}")
print(f"  Gap            : {rf_gap:.4f}\n")

print("AdaBoost")
print(f"  Acurácia treino: {ab_train_acc:.4f}")
print(f"  Acurácia teste : {ab_test_acc:.4f}")
print(f"  Gap            : {ab_gap:.4f}\n")

print("Conclusão:")
if rf_gap > ab_gap:
    print("Existe overfitting (gap treino-teste positivo). O Random Forest tende a sofrer mais neste experimento.")
elif ab_gap > rf_gap:
    print("Existe overfitting (gap treino-teste positivo). O AdaBoost tende a sofrer mais neste experimento.")
else:
    print("Ambos apresentam comportamento de overfitting semelhante neste experimento.")

Random Forest
  Acurácia treino: 1.0000
  Acurácia teste : 0.9661
  Gap            : 0.0339

AdaBoost
  Acurácia treino: 0.6730
  Acurácia teste : 0.6681
  Gap            : 0.0048

Conclusão:
Existe overfitting (gap treino-teste positivo). O Random Forest tende a sofrer mais neste experimento.


# Questão 8

Varie pelo menos um hiperparâmetro em cada modelo:

- Random Forest: `n_estimators`
- AdaBoost: `n_estimators`

## Analise:
- O desempenho muda significativamente?

## Responda:
- Qual modelo é mais sensível a mudanças?

In [23]:
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier

seed = 42
X_train, X_test, y_train, y_test = load_data(seed)

n_values = [10, 50, 100, 200]
rf_results = {}
ab_results = {}

for n in n_values:
    rf_model = RandomForestClassifier(n_estimators=n, random_state=seed)
    rf_model.fit(X_train, y_train)
    rf_results[n] = evaluate_metrics(rf_model, X_test, y_test)

    ab_model = AdaBoostClassifier(n_estimators=n, random_state=seed)
    ab_model.fit(X_train, y_train)
    ab_results[n] = evaluate_metrics(ab_model, X_test, y_test)

print("Random Forest (variação de n_estimators):")
for n in n_values:
    m = rf_results[n]
    print(f"n={n:>3} | acc={m['accuracy']:.4f} | precision={m['precision']:.4f} | recall={m['recall']:.4f} | f1={m['f1']:.4f}")

print("\nAdaBoost (variação de n_estimators):")
for n in n_values:
    m = ab_results[n]
    print(f"n={n:>3} | acc={m['accuracy']:.4f} | precision={m['precision']:.4f} | recall={m['recall']:.4f} | f1={m['f1']:.4f}")

rf_f1_values = [rf_results[n]["f1"] for n in n_values]
ab_f1_values = [ab_results[n]["f1"] for n in n_values]
rf_sensitivity = max(rf_f1_values) - min(rf_f1_values)
ab_sensitivity = max(ab_f1_values) - min(ab_f1_values)

print("\nAnálise de sensibilidade (amplitude de F1):")
print(f"Random Forest: {rf_sensitivity:.4f}")
print(f"AdaBoost:      {ab_sensitivity:.4f}")

print("\nConclusão:")
if rf_sensitivity > ab_sensitivity:
    print("O desempenho muda mais no Random Forest para os valores testados de n_estimators.")
elif ab_sensitivity > rf_sensitivity:
    print("O desempenho muda mais no AdaBoost para os valores testados de n_estimators.")
else:
    print("Os dois modelos têm sensibilidade semelhante para os valores testados de n_estimators.")

Random Forest (variação de n_estimators):
n= 10 | acc=0.9443 | precision=0.9443 | recall=0.9443 | f1=0.9442
n= 50 | acc=0.9631 | precision=0.9631 | recall=0.9631 | f1=0.9631
n=100 | acc=0.9661 | precision=0.9661 | recall=0.9661 | f1=0.9661
n=200 | acc=0.9681 | precision=0.9681 | recall=0.9681 | f1=0.9681

AdaBoost (variação de n_estimators):
n= 10 | acc=0.3337 | precision=0.2608 | recall=0.3337 | f1=0.2465
n= 50 | acc=0.6681 | precision=0.6884 | recall=0.6681 | f1=0.6711
n=100 | acc=0.7387 | precision=0.7479 | recall=0.7387 | f1=0.7412
n=200 | acc=0.7684 | precision=0.7752 | recall=0.7684 | f1=0.7698

Análise de sensibilidade (amplitude de F1):
Random Forest: 0.0239
AdaBoost:      0.5232

Conclusão:
O desempenho muda mais no AdaBoost para os valores testados de n_estimators.


**Solução**:

O desempenho muda com a variação de `n_estimators` em ambos os modelos, mas a intensidade é diferente.

- **Random Forest**: melhoria moderada (F1 de 0.9442 para 0.9681).
- **AdaBoost**: melhoria forte (F1 de 0.2465 para 0.7698).

Pela amplitude de F1 observada, o **AdaBoost é mais sensível** a mudanças em `n_estimators` neste experimento.

# Questão 9

Responda (máx. 2 parágrafos por item):

1. A acurácia é suficiente para avaliar os modelos?
2. Como você garante que o resultado não ocorreu por acaso?
3. Cite dois possíveis problemas metodológicos neste experimento.
4. O pipeline implementado é confiável? Justifique.

**1 A acurácia é suficiente para avaliar os modelos?**
Não sozinha. A acurácia resume o desempenho global, mas pode esconder erros relevantes por classe, especialmente em cenários com diferenças de dificuldade entre classes. Por isso, analisar também precisão, recall e F1-score (como foi feito) dá uma visão mais robusta do comportamento dos modelos.

**2 Como você garante que o resultado não ocorreu por acaso?**
A garantia prática vem de reprodutibilidade e análise de sensibilidade: fixar `random_state` permite repetir exatamente o experimento, e variar seeds mostra o quanto o resultado oscila sob diferentes particionamentos. Além disso, comparar múltiplas métricas e não apenas um único número reduz o risco de conclusão acidental.

**3 Cite dois possíveis problemas metodológicos neste experimento.**
(i) Avaliação baseada em um único split treino-teste pode introduzir viés de partição; uma validação cruzada estratificada daria estimativas mais estáveis.

(ii) Hiperparâmetros foram testados de forma limitada (grade pequena), o que pode subestimar o potencial de um modelo ou favorecer conclusões dependentes de escolhas pontuais.

**4 O pipeline implementado é confiável? Justifique.**
Sim, para um baseline educacional ele é confiável: há fluxo claro (carregar, treinar, avaliar), controle de aleatoriedade, comparação entre modelos e análise com diferentes seeds e hiperparâmetros. Isso atende consistência e reprodutibilidade.

Ainda assim, para maior rigor experimental, o ideal é incluir validação cruzada, análise de variância entre repetições e busca sistemática de hiperparâmetros antes de conclusões definitivas.